# 20 Newsgroups → LTS pipeline (GitHub + Colab reproducible notebook)

This notebook is organized as a reproducible Colab workflow that uses the **GitHub repository as the source of truth** for code and prompts.

Pipeline overview:

1. Mount Google Drive and clone / pull the GitHub branch  
2. Set experiment paths inside the repository  
3. Parse the raw 20 Newsgroups `.txt` source files  
4. Build the binary target dataset (`rec.autos` vs. rest)  
5. Create train / validation / test CSVs in LTS format under `data/processed/`  
6. Create the few-shot JSON under `prompts/`  
7. Verify the Python source files under `src/`  
8. Reset sampler state files before a fresh run  
9. Run `src/main_cluster.py`

This version avoids rewriting the pipeline modules inside the notebook. Instead, it expects the authoritative code to already exist in:

- `20news_rec_autos/src/`
- `20news_rec_autos/prompts/`
- `20news_rec_autos/notebooks/`

Generated CSVs are written to `data/processed/`, and generated run artifacts are written relative to the experiment folder and ignored by Git.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_URL = "https://github.com/ShenyaoZhang/DS-GA-3001-Data-Engineering-Project.git"
BRANCH = "20news-rec-autos"

DRIVE_ROOT = "/content/drive/MyDrive"
REPO_ROOT = os.path.join(DRIVE_ROOT, "DS-GA-3001-Data-Engineering-Project")
EXPERIMENT_ROOT = os.path.join(REPO_ROOT, "20news_rec_autos")

SRC_DIR = os.path.join(EXPERIMENT_ROOT, "src")
PROMPTS_DIR = os.path.join(EXPERIMENT_ROOT, "prompts")
DATA_DIR = os.path.join(EXPERIMENT_ROOT, "data")
DATA_RAW_DIR = os.path.join(DATA_DIR, "raw")
DATA_PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
OUTPUTS_DIR = os.path.join(EXPERIMENT_ROOT, "outputs")

if not os.path.exists(REPO_ROOT):
    print("Cloning repository into Google Drive...")
    !git clone -b {BRANCH} {REPO_URL} "{REPO_ROOT}"
else:
    print("Repository already exists in Drive. Pulling latest changes...")
    %cd "{REPO_ROOT}"
    !git fetch origin
    !git checkout {BRANCH}
    !git pull

%cd "{EXPERIMENT_ROOT}"

os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Experiment root:", EXPERIMENT_ROOT)
print("Raw data dir:", DATA_RAW_DIR)
print("Processed data dir:", DATA_PROCESSED_DIR)
print("Prompts dir:", PROMPTS_DIR)
print("Source dir:", SRC_DIR)
print("Files in experiment root:", sorted(os.listdir(EXPERIMENT_ROOT)))


In [ ]:
from getpass import getpass
import os

token = getpass("Enter your HF token: ")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token
print("HF token set.")

## 1) Install dependencies

This notebook installs dependencies from the repository `requirements.txt` so that the environment matches the experiment branch as closely as possible.


In [ ]:
!pip -q install -r "{os.path.join(REPO_ROOT, 'requirements.txt')}"

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2) Parse the 20 Newsgroups files

This notebook expects the raw 20 Newsgroups source files to already be in:

`20news_rec_autos/data/raw/`

with one `.txt` file per class, such as:

- `alt.atheism.txt`
- `rec.autos.txt`
- ...

Each file is parsed into records with:

`newsgroup`, `document_id`, `from`, `subject`, `body`, `text`


In [ ]:
import os
import re
import pandas as pd
import unicodedata

HEADER_KEYS = {"newsgroup", "document_id", "from", "subject"}

def _clean_field(text):
    text = str(text or "")
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\x00", " ")
    text = "".join(ch for ch in text if ch == "\n" or ch == "\t" or ch.isprintable())
    return text.strip()

def parse_newsgroup_file(filepath):
    filename = os.path.basename(filepath)
    fallback_newsgroup = filename.replace(".txt", "")

    with open(filepath, "r", encoding="latin-1", errors="ignore") as f:
        content = f.read()

    chunks = re.split(r'(?mi)(?=^Newsgroup:\s)', content)
    records = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        lines = chunk.splitlines()
        record = {
            "newsgroup": fallback_newsgroup,
            "document_id": "",
            "from": "",
            "subject": "",
            "body": "",
            "text": ""
        }

        body_start = None
        last_header_idx = -1

        for i, raw_line in enumerate(lines):
            line = raw_line.rstrip("\n")
            stripped = line.strip()

            if body_start is None and stripped == "":
                body_start = i + 1
                break

            m = re.match(r'(?i)^([A-Za-z_]+):\s*(.*)$', stripped)
            if m:
                key = m.group(1).lower()
                value = m.group(2).strip()
                if key in HEADER_KEYS:
                    record[key] = value
                    last_header_idx = i
                    continue

            # If we hit a non-header line before a blank line, treat the remainder as body
            if last_header_idx >= 0:
                body_start = i
                break

        if body_start is None:
            body_start = last_header_idx + 1 if last_header_idx >= 0 else 0

        body_lines = lines[body_start:]
        body = "\n".join(body_lines).strip()
        record["body"] = body

        if record["subject"] and record["body"]:
            record["text"] = record["subject"] + "\n\n" + record["body"]
        elif record["subject"]:
            record["text"] = record["subject"]
        else:
            record["text"] = record["body"]

        for col in ["newsgroup", "document_id", "from", "subject", "body", "text"]:
            record[col] = _clean_field(record[col])

        if record["document_id"] != "":
            records.append(record)

    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(columns=["newsgroup","document_id","from","subject","body","text"])

    df = df.drop_duplicates(subset=["document_id"]).reset_index(drop=True)
    return df

In [ ]:
txt_files = sorted([
    f for f in os.listdir(DATA_RAW_DIR)
    if f.endswith(".txt")
])

print("Found", len(txt_files), "txt files")
print(txt_files[:10])

all_dfs = []
for filename in txt_files:
    filepath = os.path.join(DATA_RAW_DIR, filename)
    file_df = parse_newsgroup_file(filepath)
    all_dfs.append(file_df)
    print(f"{filename}: {len(file_df)} documents")

raw_df = pd.concat(all_dfs, ignore_index=True)
print("\nTotal parsed rows:", len(raw_df))
print("Unique document_id:", raw_df["document_id"].nunique())
print(raw_df.head(2))


## 3) Build the binary classification dataset and a shortened `title` field

In [ ]:
TARGET_CLASS = "rec.autos"
TARGET_SLUG = TARGET_CLASS.replace(".", "_")

def looks_like_encoded_line(line):
    stripped = line.strip()
    if len(stripped) > 80 and re.fullmatch(r"[A-Za-z0-9+/=_-]+", stripped):
        return True
    if re.match(r"^(begin\s+\d{3}\s+|M[A-Z0-9`!\"#$%&'()*+,\-./:;<=>?@\[\]^_`{|}~]{20,})", stripped):
        return True
    return False

def build_llm_text(subject, body, full_text="", max_chars=1000):
    subject = _clean_field(subject)
    body = _clean_field(body)
    full_text = _clean_field(full_text)

    source_text = body if body else full_text

    cleaned_lines = []
    for line in source_text.splitlines():
        stripped = line.strip()

        if not stripped:
            continue
        if stripped.startswith(">"):
            continue
        if re.match(r"^(In article .*writes:|.*writes:)$", stripped):
            continue
        if re.match(
            r"^(From:|Subject:|Organization:|Lines:|NNTP-Posting-Host:|Reply-To:|Distribution:|Keywords:|Summary:|Archive-name:|Last-modified:|Newsgroup:|document_id:)",
            stripped,
            flags=re.IGNORECASE
        ):
            continue
        if re.fullmatch(r"[-_=~*`]{4,}", stripped):
            continue
        if looks_like_encoded_line(stripped):
            continue

        cleaned_lines.append(stripped)

    cleaned_body = " ".join(cleaned_lines)
    cleaned_body = re.sub(r"\s+", " ", cleaned_body).strip()

    if subject and cleaned_body:
        combined = f"{subject}\n\n{cleaned_body}"
    elif subject:
        combined = subject
    elif cleaned_body:
        combined = cleaned_body
    else:
        combined = full_text[:max_chars]

    return combined[:max_chars]

task_df = raw_df.copy()
task_df["binary_label"] = (task_df["newsgroup"] == TARGET_CLASS).astype(int)

task_df["raw_subject"] = task_df.get("subject", "").fillna("").astype(str)
task_df["raw_body"] = task_df.get("body", "").fillna("").astype(str)
task_df["raw_text"] = task_df.get("text", "").fillna("").astype(str)

task_df["llm_text"] = task_df.apply(
    lambda r: build_llm_text(
        r.get("subject", ""),
        r.get("body", ""),
        r.get("text", ""),
        max_chars=1000
    ),
    axis=1
)

# fallback if any rows still ended up empty
empty_mask = task_df["llm_text"].str.strip() == ""
task_df.loc[empty_mask, "llm_text"] = (
    task_df.loc[empty_mask, "raw_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 1000)
)

# make title the one text column used by the pipeline
task_df["title"] = task_df["llm_text"]

dedup_df = (
    task_df.groupby("document_id", as_index=False)
           .agg({
               "raw_text": "first",
               "raw_subject": "first",
               "raw_body": "first",
               "title": "first",
               "newsgroup": "first",
               "binary_label": "max"
           })
)

print("Deduplicated shape:", dedup_df.shape)
print(dedup_df["binary_label"].value_counts())
print(dedup_df["binary_label"].value_counts(normalize=True))
print(dedup_df[["newsgroup", "binary_label", "raw_subject", "title"]].head(3))

In [ ]:
train_df, test_df = train_test_split(
    dedup_df,
    test_size=0.20,
    stratify=dedup_df["binary_label"],
    random_state=SEED
)

train_inner_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["binary_label"],
    random_state=SEED
)

train_inner_df = train_inner_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train_inner:", train_inner_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

In [ ]:
import csv

def to_lts_format(df):
    out = df.rename(columns={
        "document_id": "id",
        "binary_label": "label",
        "raw_subject": "subject",
    }).copy()

    out["title"] = out["title"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    out["subject"] = out["subject"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

    keep_cols = ["id", "title", "subject", "label", "newsgroup"]
    return out[keep_cols]

train_inner_lts = to_lts_format(train_inner_df)
val_lts = to_lts_format(val_df)
test_lts = to_lts_format(test_df)

train_inner_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_20news_{TARGET_SLUG}.csv")
val_path = os.path.join(DATA_PROCESSED_DIR, f"val_20news_{TARGET_SLUG}.csv")
test_path = os.path.join(DATA_PROCESSED_DIR, f"test_20news_{TARGET_SLUG}.csv")

train_inner_lts.to_csv(train_inner_path, index=False, quoting=csv.QUOTE_ALL)
val_lts.to_csv(val_path, index=False, quoting=csv.QUOTE_ALL)
test_lts.to_csv(test_path, index=False, quoting=csv.QUOTE_ALL)

print(train_inner_path)
print(val_path)
print(test_path)
print(train_inner_lts["label"].value_counts())
print(train_inner_lts[["id", "newsgroup", "label", "subject", "title"]].head(2))


## 4) Build a curated few-shot prompt set for the labeler (starter template for `rec.autos`)

This cell now writes a safe starter JSON for `rec.autos`. After you inspect the new train split, we can choose good positive and negative IDs together.

In [ ]:
# Curated few-shot examples for rec.autos.
# This restores the exact working IDs/order from the earlier successful file.

CURATED_POSITIVE_IDS = [
    101558,  # 1993 Infiniti G20
    101637,  # Jeep Grand vs. Toyota 4-Runner
    103154,  # locking lugnuts / tire rebalance
    103176,  # Slick 50, any good?
    103212,  # Honda Mailing list?
    103812,  # Changing oil by yourself
    102837,  # Taurus/Sable rotor recall
    103377,  # legal car buying problems
]

CURATED_NEGATIVE_IDS = [
    104698,  # rec.motorcycles BMW mailing list
    104882,  # rec.motorcycles carb discussion
    104795,  # rec.motorcycles countersteering
    74810,   # misc.forsale car sale post
    39024,   # comp.graphics VESA driver
    38348,   # comp.graphics VGA/SVGA programming
]

def _clean_text(text, max_chars=140):
    return str(text).strip().replace("\n", " ")[:max_chars]

def build_curated_few_shot(df, positive_ids, negative_ids, id_col="id", text_col="title", label_col="label"):
    work = df.copy()
    work[id_col] = work[id_col].astype(str)
    chosen_ids = [str(x) for x in (positive_ids + negative_ids)]
    chosen = work[work[id_col].isin(chosen_ids)].copy()

    examples = []
    missing_ids = []

    for doc_id in chosen_ids:
        row = chosen[chosen[id_col] == doc_id]
        if row.empty:
            missing_ids.append(doc_id)
            continue

        row = row.iloc[0]
        doc_id_int = int(row[id_col])

        examples.append({
            "id": doc_id_int,
            "text": _clean_text(row[text_col], max_chars=140),
            "label": str(int(row[label_col]))
        })

    if missing_ids:
        print("Missing curated IDs from train split:", missing_ids)

    return examples

few_shot_examples = build_curated_few_shot(
    train_inner_lts,
    CURATED_POSITIVE_IDS,
    CURATED_NEGATIVE_IDS,
    id_col="id",
    text_col="title",
    label_col="label"
)

few_shot_path = os.path.join(PROMPTS_DIR, f"few_shot_examples_{TARGET_SLUG}.json")

with open(few_shot_path, "w", encoding="utf-8") as f:
    json.dump(few_shot_examples, f, indent=2, ensure_ascii=False)

print("Saved few-shot examples to:", few_shot_path)
print("Number of curated examples:", len(few_shot_examples))
few_shot_examples[:5] if few_shot_examples else []

## 5) Use the Python source files from the repository

This notebook does **not** rewrite the pipeline modules. Instead, it uses the authoritative source files stored in:

- `20news_rec_autos/src/`
- `20news_rec_autos/prompts/`

The next cells verify that those files exist before running the experiment.


In [ ]:
EXPECTED_SRC_FILES = [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "main_cluster.py",
]

missing = []
for fname in EXPECTED_SRC_FILES:
    fpath = os.path.join(SRC_DIR, fname)
    exists = os.path.exists(fpath)
    print(f"{fname}: {'FOUND' if exists else 'MISSING'}")
    if not exists:
        missing.append(fname)

if missing:
    raise FileNotFoundError(f"Missing source files in src/: {missing}")
else:
    print("\nAll required source files are present.")


In [ ]:
few_shot_repo_path = os.path.join(PROMPTS_DIR, f"few_shot_examples_{TARGET_SLUG}.json")
print("Few-shot prompt file:", few_shot_repo_path)
print("Exists?", os.path.exists(few_shot_repo_path))

if os.path.exists(few_shot_repo_path):
    with open(few_shot_repo_path, "r", encoding="utf-8") as f:
        few_shot_preview = json.load(f)
    print("Number of examples:", len(few_shot_preview))
    print("First two examples:")
    print(few_shot_preview[:2])


## 6) Reset state files before a fresh run

This step is important.

The pipeline stores root-level sampler state files such as:
- `selected_ids.txt`
- `wins.txt`
- `losses.txt`
- `positive_data.csv`

It also writes generated run artifacts to `data/processed/`, such as:
- `*_model_results.json`
- `*_training_data.csv`
- `*_data_labeled.csv`
- `*_lda.csv`

If you rerun without clearing them, sampling may behave strangely or appear to reuse old state.


In [ ]:
for fname in [
    "selected_ids.txt",
    "wins.txt",
    "losses.txt",
    "positive_data.csv",
]:
    path = os.path.join(EXPERIMENT_ROOT, fname)
    if os.path.exists(path):
        os.remove(path)
        print("Removed:", path)

for fname in [
    f"train_inner_20news_{TARGET_SLUG}_model_results.json",
    f"train_inner_20news_{TARGET_SLUG}_training_data.csv",
    f"train_inner_20news_{TARGET_SLUG}_data_labeled.csv",
    f"train_inner_20news_{TARGET_SLUG}_lda.csv",
]:
    path = os.path.join(DATA_PROCESSED_DIR, fname)
    if os.path.exists(path):
        os.remove(path)
        print("Removed:", path)


## 7) Sanity-check the Python files

This catches syntax problems before the long run starts.

In [ ]:
import subprocess

compile_targets = [os.path.join(SRC_DIR, f) for f in [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "main_cluster.py",
]]

result = subprocess.run(
    ["python", "-m", "py_compile", *compile_targets],
    capture_output=True,
    text=True,
    cwd=EXPERIMENT_ROOT,
)

if result.returncode == 0:
    print("All files compiled successfully.")
else:
    print("Compilation failed:")
    print(result.stderr)


In [ ]:
%cd "{EXPERIMENT_ROOT}"

This version keeps only the columns `id`, `title`, `subject`, `label`, and `newsgroup` in the generated train/val/test CSVs. The pipeline uses the shortened cleaned `title` text for clustering, Qwen labeling, and BERT fine-tuning. The generated CSVs live under `data/processed/`, while the prompt file is read from `prompts/`. The run below is configured for `rec.autos`.


In [ ]:
!python src/main_cluster.py \
  -sample_size 150 \
  -filename "data/processed/train_inner_20news_rec_autos" \
  -val_path "data/processed/val_20news_rec_autos.csv" \
  -balance False \
  -sampling "thompson" \
  -filter_label True \
  -model_finetune "bert-base-uncased" \
  -labeling "qwen" \
  -model "text" \
  -baseline 0.5 \
  -metric "f1_pos" \
  -cluster_size 10 \
  -target_class "rec.autos" \
  -few_shot_path "prompts/few_shot_examples_rec_autos.json" \
  -hf_model_id "Qwen/Qwen2.5-3B-Instruct" \
  -max_iterations 4


In [ ]:
import pandas as pd

labeled_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_20news_{TARGET_SLUG}_data_labeled.csv")
df = pd.read_csv(labeled_path)
print(df["label"].value_counts(dropna=False))

if "true_label" in df.columns:
    print(df["true_label"].value_counts(dropna=False))
    print("Agreement:", (df["label"] == df["true_label"]).mean())

print(df[["title", "label"]].head(10))


In [ ]:
clustered_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_20news_{TARGET_SLUG}_lda.csv")
clustered_df = pd.read_csv(clustered_path)
print(clustered_df.columns.tolist())
print(clustered_df[["id", "newsgroup", "label", "label_cluster", "title"]].head(10).to_string(index=False))


In [ ]:
cluster_summary = (
    clustered_df.groupby("label_cluster")
    .agg(
        cluster_size=("label", "size"),
        num_positive=("label", "sum")
    )
)
cluster_summary["positive_rate"] = cluster_summary["num_positive"] / cluster_summary["cluster_size"]
print(cluster_summary.sort_values("positive_rate", ascending=False))

## 9) Summarize final run outputs

Use this cell after the run finishes to inspect the saved evaluation metrics and cluster composition.


In [ ]:
import json
import pandas as pd

results_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_20news_{TARGET_SLUG}_model_results.json")
if os.path.exists(results_path):
    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    print("Saved model results keys:", list(results.keys()))
    flat = []
    for k, vals in results.items():
        for v in vals:
            row = {"run_key": k}
            row.update(v)
            flat.append(row)
    results_df = pd.DataFrame(flat)
    display(results_df)
else:
    print("No model results json found:", results_path)

labeled_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_20news_{TARGET_SLUG}_data_labeled.csv")
if os.path.exists(labeled_path):
    labeled_df = pd.read_csv(labeled_path)
    print("\nPseudo-label distribution:")
    print(labeled_df["label"].value_counts(dropna=False))
    if "true_label" in labeled_df.columns:
        print("\nTrue-label distribution:")
        print(labeled_df["true_label"].value_counts(dropna=False))
        print("Agreement:", (labeled_df["label"] == labeled_df["true_label"]).mean())
else:
    print("No labeled csv found:", labeled_path)
